In [1]:
import timeit
import cv2
import numba
from numba import cuda
import numpy as np

In [2]:
@cuda.jit
def gpu_noise_reduction(input_img, output_img) -> None:
    """
    Ядро функции для применения медианного фильтрa к изображению.

    Args:
        input_img: Исходное изображение.
        output_img: Обработанное изображение.
    """
    idx = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    height, width = input_img.shape[0] - 2, input_img.shape[1] - 2
    while idx < height * width:
        pixel_window = cuda.local.array(shape=9, dtype=numba.float64)
        row, col = idx // width, idx % width
        for n in range(9):
            pixel_window[n] = input_img[row + n // 3, col + n % 3]
        for n in range(8):
            for m in range(8-n):
                if pixel_window[m] > pixel_window[m + 1]:
                    pixel_window[m], pixel_window[m + 1] = pixel_window[m + 1], pixel_window[m]
        output_img[row, col] = pixel_window[4]
        idx += cuda.blockDim.x * cuda.gridDim.x

In [3]:
def cpu_noise_reduction(input_img: np.ndarray) -> np.ndarray:
    """
    Применение медианного фильтра к изображению на CPU.

    Args:
        input_img: Исходное изображение.
    Outs:
        Обработанное изображение.
    """
    height, width = input_img.shape[0] - 2, input_img.shape[1] - 2
    output_img = np.zeros((height, width))
    for row in range(height):
        for col in range(width):
            pixel_window = input_img[row:row+3, col:col+3].flatten()
            pixel_window.sort()
            output_img[row, col] = pixel_window[4]
    return output_img

In [4]:
def execute_gpu_filter(input_img: np.ndarray, output_img: np.ndarray) -> np.ndarray:
    """
    Применение медианного фильтра с использованием CUDA.

    Args:
        input_img: Исходное изображение.
        output_img: Обработанное изображение.
    Out:
        Обработанное изображение.
    """
    # Загружаем данные в память GPU
    device_input_img = cuda.to_device(input_img)
    device_output_img = cuda.device_array_like(output_img)

    # Параметры запуска CUDA
    threads_per_block = cuda.get_current_device().WARP_SIZE
    blocks_per_grid = 1024

    # Вызов CUDA-ядра
    gpu_noise_reduction[blocks_per_grid, threads_per_block](
        device_input_img, device_output_img)
    return device_output_img.copy_to_host()

In [8]:
noise_prob = 0.3
original_img = cv2.imread('main.jpg', cv2.IMREAD_GRAYSCALE)
noisy_img = np.where(np.random.rand(
    *original_img.shape) < noise_prob, np.random.randint(0, 255, original_img.shape), original_img)
cv2.imwrite('noisy_image.jpg', noisy_img)

test_count = 12
padded_img = np.pad(noisy_img, 1, mode="symmetric")
filtered_img = np.zeros(noisy_img.shape)

gpu_result = execute_gpu_filter(padded_img, filtered_img)
cpu_result = cpu_noise_reduction(padded_img)

print(np.allclose(cpu_result, gpu_result))
cv2.imwrite('filtered_gpu.jpg', gpu_result)
cv2.imwrite('filtered_cpu.jpg', cpu_result)

gpu_time = timeit.timeit(lambda: execute_gpu_filter(
    padded_img, filtered_img), number=test_count) / test_count
cpu_time = timeit.timeit(lambda: cpu_noise_reduction(
    padded_img), number=test_count) / test_count

print(f"GPU Execution Time = {gpu_time}")
print(f"CPU Execution Time = {cpu_time}")
print(f"Performance Improvement = {cpu_time / gpu_time}")

True
GPU Execution Time = 0.00459498899999744
CPU Execution Time = 1.5239687362499978
Performance Improvement = 331.6588431987208
